# Independent Clean CV-DV LCHS Demo

This notebook imports only the new `clean_*` modules. It treats the working paper as a hypothesis rather than ground truth and checks the implementation against exact linear algebra wherever possible.

Sections:
- Dirichlet heat-equation benchmark
- convention and Pauli reconstruction checks
- coefficient backend agreement
- exact-vs-Trotter validation
- state-prep/readout comparison across `injection`, `snap_d`, and `givens`
- generic Pauli-system template

In [ ]:
import numpy as np

from clean_core import (
    EvolutionSpec,
    KernelSpec,
    StatePrepSpec,
    build_dirichlet_heat_system,
    build_pauli_system,
    coefficient_backend_gap,
    exact_reference_map,
    generator_matrix,
)
from clean_hybrid import run_clean_lchs

## 1. Dirichlet Heat Equation Setup

The clean implementation uses internal `hbar=1` conventions and reconstructs the heat operator directly from the semidiscrete tridiagonal Laplacian before decomposing it into Pauli terms.

In [ ]:
num_qubits = 2
alpha = 1.0
grid_spacing = 1.0
total_time = 1.0
init_basis_index = 1

system = build_dirichlet_heat_system(
    num_qubits=num_qubits,
    alpha=alpha,
    grid_spacing=grid_spacing,
    total_time=total_time,
    init_basis_index=init_basis_index,
)

A = generator_matrix(system)
print('System label:', system.label)
print('Generator matrix A = L + iH:')
print(A)
print('\nExact reference map exp(-AT):')
print(exact_reference_map(system))
print('\nPauli decomposition (L block):')
for term in system.l_terms:
    print(f'  {term.label}: {term.coeff.real:+.8f}{term.coeff.imag:+.8f}j')

## 2. Convention and Coefficient Checks

In [ ]:
kernel = KernelSpec(
    r_target=1.1,
    r_prime=0.1,
    beta=0.75,
    n_coeff=12,
    n_fock=32,
    n_quad=180,
    coeff_backend='explicit_overlap',
)

print('Coefficient backend gap (explicit vs Gauss-Hermite):', coefficient_backend_gap(kernel))
print('Internal convention: hbar=1, x_hat=(a+a^dagger)/sqrt(2)')

## 3. State Preparation and Readout Comparison

The three paths are:
- `injection`: exact simulator seed loading
- `snap_d`: gate-level SNAP + displacement synthesis
- `givens`: analytic Law-Eberly / Givens angles with direct injection in simulation

In [ ]:
methods = ['injection', 'snap_d', 'givens']
evolution = EvolutionSpec(n_trotter_steps=20, readout_mode='postselect_statevector')

results = {}
for method in methods:
    prep = StatePrepSpec(method=method, snap_depth=3, snap_restarts=2, snap_maxiter=300)
    results[method] = run_clean_lchs(system, kernel, prep, evolution)

for method, result in results.items():
    print(f'[{method}] fidelity_vs_exact={result.fidelity_vs_exact:.8f}  '
          f'post_prob={result.postselection_probability:.6e}  '
          f'oracle_fidelity={result.oracle_fidelity:.8f}  '
          f'depth={result.circuit_depth}')

## 4. Exact-vs-Trotter Comparison

In [ ]:
trotter_grid = [5, 10, 20, 40]
prep = StatePrepSpec(method='injection')
for steps in trotter_grid:
    evo = EvolutionSpec(n_trotter_steps=steps, readout_mode='postselect_statevector')
    res = run_clean_lchs(system, kernel, prep, evo)
    print(f'n_trotter_steps={steps:3d}  fidelity_vs_exact={res.fidelity_vs_exact:.8f}  '
          f'rel_error_vs_truncated={res.rel_error_vs_truncated:.6e}')

## 5. Generic Pauli-System Template

Edit `l_terms`, `h_terms`, `total_time`, and `init_state` below to test a different model without touching the implementation modules.

In [ ]:
generic_system = build_pauli_system(
    l_terms=[('I', 0.15 + 0.0j)],
    h_terms=[('Y', -0.8 + 0.0j)],
    total_time=0.7,
    init_state=np.array([1.0, 1.0j]),
    label='generic_single_qubit_demo',
)
generic_kernel = KernelSpec(
    r_target=1.0,
    r_prime=0.15,
    beta=0.7,
    n_coeff=10,
    n_fock=32,
    n_quad=160,
)
generic_prep = StatePrepSpec(method='injection')
generic_evo = EvolutionSpec(n_trotter_steps=20, readout_mode='postselect_statevector')
generic_result = run_clean_lchs(generic_system, generic_kernel, generic_prep, generic_evo)
print('Generic system fidelity_vs_exact:', generic_result.fidelity_vs_exact)
print('Generic system postselection_probability:', generic_result.postselection_probability)
print('Generic system rel_error_vs_exact:', generic_result.rel_error_vs_exact)